In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import numpy as np
import sklearn.manifold, sklearn.cluster
import rdkit, rdkit.Chem, rdkit.Chem.Draw
from rdkit.Chem.Draw import IPythonConsole
np.random.seed(0)
import warnings
IPythonConsole.ipython_useSVG = True
warnings.filterwarnings('ignore')
sns.set_context('notebook')
sns.set_style('white',  {'xtick.bottom':True, 'ytick.left':True, 'xtick.color': '#666666', 'ytick.color': '#666666',
                        'axes.edgecolor': '#666666', 'axes.linewidth':     0.8 })
color_cycle = ['#1bbc28', '#F06060', '#5C4B51', '#F3B562', '#6e5687']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=color_cycle) 

In [ ]:
soldata = pd.read_csv('../data/curated-solubility-dataset.csv')
soldata.head()
print (len(soldata))

In [ ]:

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, Descriptors3D, Draw, rdMolDescriptors, Draw, PandasTools
from rdkit.DataManip.Metric.rdMetricMatrixCalc import GetTanimotoSimMat, GetTanimotoDistMat
from rdkit.Chem.Draw import IPythonConsole

%config Completer.use_jedi = False
PandasTools.RenderImagesInAllDataFrames(images=True)

In [ ]:
mols = [Chem.MolFromSmiles(smi) for smi in soldata.SMILES]

In [ ]:
#bar=progressbar.ProgressBar(max_value=len(soldata))
table=pd.DataFrame()
for i,mol in enumerate(mols):
    #Chem.SanitizeMol(mol)
    soldata.loc[i,'SMILES']=Chem.MolToSmiles(mol)
    soldata.loc[i,'Mol']=mol
    soldata.loc[i,'NumAliphaticCarbocycles']=Descriptors.NumAliphaticCarbocycles(mol)
    soldata.loc[i,'NumAliphaticHeterocycles']=Descriptors.NumAliphaticHeterocycles(mol)
    soldata.loc[i,'NumAliphaticRings']=Descriptors.NumAliphaticRings(mol)
    soldata.loc[i,'NumAromaticCarbocycles']=Descriptors.NumAromaticCarbocycles(mol)
    soldata.loc[i,'NumAromaticHeterocycles']=Descriptors.NumAromaticHeterocycles(mol)
    soldata.loc[i,'FractionCSP3']=Descriptors.FractionCSP3(mol)
    #bar.update(i+1)

In [ ]:
first_column = soldata.pop('Mol')
soldata.insert(0, 'Mol', first_column)
soldata.head(5)

In [ ]:
soldata.columns

In [ ]:
features_start_at = list(soldata.columns).index('MolWt')
feature_names = soldata.columns[features_start_at:]

fig, axs = plt.subplots(nrows=7, ncols=4, sharey=True, figsize=(12, 8), dpi=400)
axs = axs.flatten() # don't want to think about i/j
for i,n in enumerate(feature_names):
    ax = axs[i]
    ax.scatter(
        soldata[n], soldata.Solubility, 
        s = 6, alpha=0.4,
        color = f'C{i}') # add some color 
    if i % 4 == 0:
        ax.set_ylabel('Solubility')
    ax.set_xlabel(n)
# hide empty subplots
for i in range(len(feature_names), len(axs)):
    fig.delaxes(axs[i])
plt.tight_layout()
plt.show()

# Data cleaning

In [ ]:
soldata.drop(soldata[soldata['MolWt'] < 120].index, inplace = True)  or soldata.drop(soldata[soldata['MolWt'] > 600].index, inplace = True)
soldata.drop(soldata[soldata['TPSA'] < 30].index, inplace = True)  or soldata.drop(soldata[soldata['TPSA'] > 140].index, inplace = True)
soldata.drop(soldata[soldata['NumRotatableBonds'] > 12].index, inplace = True)

In [ ]:
len(soldata)

# after removing outliers

In [ ]:
features_start_at = list(soldata.columns).index('MolWt')
feature_names = soldata.columns[features_start_at:]

fig, axs = plt.subplots(nrows=7, ncols=4, sharey=True, figsize=(12, 8), dpi=400)
axs = axs.flatten() # don't want to think about i/j
for i,n in enumerate(feature_names):
    ax = axs[i]
    ax.scatter(
        soldata[n], soldata.Solubility, 
        s = 6, alpha=0.4,
        color = f'C{i}') # add some color 
    if i % 4 == 0:
        ax.set_ylabel('Solubility')
    ax.set_xlabel(n)
# hide empty subplots
for i in range(len(feature_names), len(axs)):
    fig.delaxes(axs[i])
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

bins = [np.log10(x*1e-6) for x in [30,200]]
bins = [-100] + bins + [100]
soldata['bin'] = pd.cut(soldata.Solubility,bins=bins,labels=["Low","Medium","High"])

In [ ]:
soldata.bin

In [ ]:
value=[]
for i,row in soldata.iterrows():
    if row['bin'] == 'Low':
        value.append(0)
    else:
        value.append(1)

In [ ]:
len(value)

In [ ]:
soldata['Value'] = value

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem.Draw import SimilarityMaps

def genFP(mol,Dummy=-1):
    fp = SimilarityMaps.GetMorganFingerprint(mol)   #calulates the atom pairs finger  prints
    fp_vect = np.zeros((1))
     
    DataStructs.ConvertToNumpyArray(fp, fp_vect)
    return fp_vect

In [ ]:
mols=[]
X=[]
y=[]

for i,row in soldata.iterrows():
    try:
        mol = Chem.MolFromSmiles(row['SMILES'])
        if type(mol)!=type(None):
            fp_vect=genFP(mol)
            mols.append(mol)
            
            X.append(fp_vect)
            y.append(row['Value'])
            
    except:
        print("failed")
        
print("smiles",len(soldata))
print("converted",len(mols))

# Model building

In [ ]:
from sklearn import linear_model
import sklearn.model_selection

In [ ]:
X=np.array(X)
y=np.array(y)

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, train_size=0.8, test_size=0.2)

In [ ]:
# Dependencies are pinned in ../requirements.txt — skip this cell if already installed.
# %pip install xgboost


In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier()

In [ ]:
xgb.fit(X_train,y_train)

In [ ]:
xgb.score(X_train,y_train)

In [ ]:
xgb.score(X_test,y_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
X_test

In [ ]:
predict = xgb.predict(X_test)

In [ ]:
confusion_matrix(y_test,predict)

In [ ]:
print(classification_report(y_test,predict))

# Similarity maps

In [ ]:
def getProba(fp, predictionFunction):
  return predictionFunction((fp,))[0][1]
mols = [(Chem.MolFromSmiles('OCC(O)CO'),'Glycerol'),
    (Chem.MolFromSmiles('c1ccc2c(c1)cc3ccc4cccc5c4c3c2cc5'),'Benzo[a]pyrene'),
    (Chem.MolFromSmiles('CCN(N=O)C(N)=O'),'N-ethyl-N-nitrosourea'),
    (Chem.MolFromSmiles('CC(C)CCC(C)CCC1=CC=C2C(=C1)C=CC=C2[N+](=O)[O-]'),'Tester')
    ]


In [ ]:
for mol,name in mols:
    #Make Prediction
    fp_vect = genFP(mol)
    fp_vect =fp_vect.reshape(1, -1)
    print ('Probability %s mutagenic %0.2f'%(name,estimator.predict_proba(fp_vect)[0][0]))

In [ ]:
weights = SimilarityMaps.GetAtomicWeightsForModel(mol, SimilarityMaps.GetMorganFingerprint, lambda x: getProba(x, estimator.predict_proba))
fig = SimilarityMaps.GetSimilarityMapFromWeights(mol, weights,colorMap='coolwarm')
fig.savefig('similarity.png', bbox_inches='tight')

# saving the model using pickle

# Predictions

In [ ]:
df = pd.read_excel('../data/External_set.xlsx')

In [ ]:
df

In [ ]:
mols1=[]
fp=[]
for i,row in df.iterrows():
    try:
        mol = Chem.MolFromSmiles(row['SMILES'])
        if type(mol)!=type(None):
            fp_vect=genFP(mol)
            mols1.append(mol)
            
            fp.append(fp_vect)
        else:
            print(i)

            
    except:
        print("failed")

There was a invalid smile so i just dropped using the index

In [ ]:
df.iloc[54]

In [ ]:
updtd_df = df.drop(54)

In [ ]:
len(updtd_df)

In [ ]:
molss=[]
fpp=[]
for i,row in updtd_df.iterrows():
    try:
        mol = Chem.MolFromSmiles(row['SMILES'])
        if type(mol)!=type(None):
            fp_vect=genFP(mol)
            molss.append(mol)
            
            fpp.append(fp_vect)
        else:
            print(i)

            
    except:
        print("failed")

In [ ]:
predict = xgb.predict(fpp)

In [ ]:
predict

In [ ]:
y_test

In [ ]:
from sklearn.metrics import cohen_kappa_score

In [ ]:
predictt = xgb.predict(X_test)

In [ ]:
cohen_kappa_score(y_test, predictt)

my cohens kappa score got a moderate score so my model works good